# # Film Eleştirileri ve Bag-of-Words Modellemesi

🎯 Bu zorluğun amacı, metinlerin ***Bag-of-words*** modellemesiyle oynamaktır.

✍️ Aşağıdaki veri setinde, _“olumlu”_ veya _“olumsuz”_ olarak sınıflandırılmış 2000 adet yorum bulunmaktadır.

In [1]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/movie_reviews.csv")
data.head()

,target,reviews
0,neg,"plot : two teen couples go to a church party ,..."
1,neg,the happy bastard's quick movie review \ndamn ...
2,neg,it is movies like these that make a jaded movi...
3,neg,""" quest for camelot "" is warner bros . ' firs..."
4,neg,synopsis : a mentally unstable man undergoing ...


In [2]:
data.shape

(2000, 2)

## 1. Ön işleme (Preprocessing)

❓ **Soru (Metin Temizleme)** ❓

- Bir cümleyi temizleyecek bir `preprocessing` fonksiyonu yazın ve bunu tüm yorumlara uygulayın. Fonksiyon şunları yapmalıdır:
    - boşlukları kaldırma
    - harfleri küçük harfe çevirme
    - sayıları kaldırma
    - noktalama işaretlerini kaldırma
    - tokenization (kelimelere ayırma)
    - lemmatization (kelime köküne indirgeme)
- Temizlenmiş yorumları `clean_reviews` adlı bir sütunda saklayabilirsiniz.
- Bu aşamada stopword’leri kaldırmayın; nedenini `3. N-gram modelleme` bölümünde açıklayacağız.

In [3]:
import string
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer

def preprocessing(sentence):
    # 1. Kenar boşluklarını kaldırma (strip)
    sentence = sentence.strip()
    
    # 2. Küçük harfe çevirme
    sentence = sentence.lower()
    
    # 3. Sayıları kaldırma
    sentence = "".join([char for char in sentence if not char.isdigit()])
    
    # 4. Noktalama işaretlerini kaldırma
    sentence = "".join([char for char in sentence if char not in string.punctuation])
    
    # 5. Tokenization (Kelimelere ayırma)
    tokens = word_tokenize(sentence)
    
    # 6. Lemmatization (Köklerine ayırma)
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    # Sonuç bir liste değil, tek bir dize (string) olmalı
    return " ".join(lemmatized_tokens)

In [4]:
# Yorumları temizle
data['clean_reviews'] = data['reviews'].apply(preprocessing)

❓ **Soru (LabelEncoding)**❓

Hedefinizi LabelEncode ile kodlayın ve `“target_encoded”` adlı bir sütuna kaydedin.

In [5]:
from sklearn.preprocessing import LabelEncoder

# 1. LabelEncoder nesnesini oluşturalım
le = LabelEncoder()

# 2. 'target' sütununu (neg/pos) sayılara dönüştürüp yeni sütuna kaydedelim
data['target_encoded'] = le.fit_transform(data['target'])

# 3. İlk birkaç satırı kontrol edelim
data[['target', 'target_encoded']].head()

,target,target_encoded
0,neg,0
1,neg,0
2,neg,0
3,neg,0
4,neg,0


In [6]:
# Hızlı kontrol
data.head()

,target,reviews,clean_reviews,target_encoded
0,neg,"plot : two teen couples go to a church party ,...",plot two teen couple go to a church party drin...,0
1,neg,the happy bastard's quick movie review \ndamn ...,the happy bastard quick movie review damn that...,0
2,neg,it is movies like these that make a jaded movi...,it is movie like these that make a jaded movie...,0
3,neg,""" quest for camelot "" is warner bros . ' firs...",quest for camelot is warner bros first feature...,0
4,neg,synopsis : a mentally unstable man undergoing ...,synopsis a mentally unstable man undergoing ps...,0


## 2. Bag-of-Words Modellemesi

❓ **Soru (Tek kelimelik sözcüklerle NaiveBayes)** ❓

`cross_validate` kullanarak, metinlerin Bag-of-Words temsilinde eğitilmiş bir Multinomial Naive Bayes modelini puanlayın.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_validate

# 1. Bag-of-Words (Unigram) vektörleştirme
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(data['clean_reviews'])
y = data['target_encoded']

# 2. Multinomial Naive Bayes modelini tanımlama
nb_model = MultinomialNB()

# 3. cross_validate ile modelin puanlanması (5 katlı doğrulama)
cv_results = cross_validate(nb_model, X_bow, y, cv=5, scoring='accuracy')

# 4. Sonuçları yazdırma
print(f"Katman Skorları: {cv_results['test_score']}")
print(f"Ortalama Doğruluk: {cv_results['test_score'].mean():.4f}")

Katman Skorları: [0.805  0.82   0.805  0.835  0.8175]
Ortalama Doğruluk: 0.8165


## 3. N-gram Modellemesi

👀 Stop kelimeleri kaldırmamanızı istediğimizi hatırlayın. Neden? 

👉 Naive Bayes modelini bigramlarla eğiteceğiz. Bu nedenle, “I do not like coriander” (Kişnişi sevmiyorum) gibi bir cümlede, örneğin bu cümlede olumsuzluğu tespit etmek için “do not” bigramını taramak önemlidir.

❓ **Soru (bigramlarla NaiveBayes)** ❓

`cross_validate` kullanarak, metinlerin 2-gram Bag-of-Words temsilinde eğitilmiş bir Multinomial Naive Bayes modelini puanlayın.

In [8]:
vectorizer = CountVectorizer(ngram_range = (2,2))
naivebayes = MultinomialNB()

X_bow = vectorizer.fit_transform(data.clean_reviews)

cv_nb = cross_validate(
    naivebayes,
    X_bow,
    data.target_encoded,
    scoring = "accuracy"
)

round(cv_nb['test_score'].mean(),2)

0.84

🏁 Tebrikler! Artık vektörleştirilmiş metinler üzerinde Naive Bayes modelini nasıl eğiteceğinizi biliyorsunuz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

